<a href="https://colab.research.google.com/github/knyazevaanna16-arch/programming_tsib_251_12/blob/main/lab_04_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# task_04_02_12

# Домен / Сфера: Клиника
# Дочерние классы 1 (Наследуют БК 1): Consultation (врач), LabTest (тип анализа)
# Дочерний класс 2 (Наследует БК 2): InsuredPatient (страх. полис)
# Контейнер: Visit
# Полиморфный метод (Бизнес-логика): calculate_final_bill() (скидка по страховке)

# Выполнил: Князева А.П.
# Группа ЦИБ 251

# ==============================================================================
# БАЗОВЫЕ КЛАССЫ (из ЛР 4.1, с небольшими доработками для ЛР 4.2)
# ==============================================================================

class MedicalService:
    """Базовый класс 1: Медицинская услуга (Сущность)."""

    def __init__(self, name, cost):
        self.name = name
        self.cost = cost

    def calculate_final_bill(self):
        """
        Полиморфный метод.
        В базовом классе возвращает полную стоимость.
        Переопределяется в дочерних классах.
        """
        return self.cost

    def __str__(self):
        return f"Услуга: {self.name:<20} | Базовая цена: {self.cost} руб."

class Patient:
    """Базовый класс 2: Пациент (Субъект)."""

    def __init__(self, full_name, card_number):
        self.full_name = full_name
        self.card_number = card_number

    def __str__(self):
        return f"Пациент: {self.full_name:<25} | Карта №: {self.card_number}"

# ==============================================================================
# ЭТАП 1 & 2: НАСЛЕДОВАНИЕ И ПОЛИМОРФИЗМ (Дочерние классы 1)
# ==============================================================================

class Consultation(MedicalService):
    """
    Дочерний класс 1: Консультация врача.
    Наследует MedicalService.
    Специфичное поле: doctor_name (имя врача).
    """

    def __init__(self, name, cost, doctor_name):
        # Используем super() для инициализации общих полей (name, cost)
        super().__init__(name, cost)
        self.doctor_name = doctor_name

    def calculate_final_bill(self):
        """
        Переопределение бизнес-логики.
        Для консультаций может применяться своя логика, например,
        фиксированная цена без изменений или небольшая наценка за срочность.
        В данном примере оставляем как есть, но метод переопределен.
        """
        return self.cost

    def __str__(self):
        base_str = super().__str__()
        return f"{base_str} | Врач: {self.doctor_name}"

class LabTest(MedicalService):
    """
    Дочерний класс 1: Лабораторный анализ.
    Наследует MedicalService.
    Специфичное поле: test_type (тип анализа, например, 'Общий', 'Биохимия').
    """

    def __init__(self, name, cost, test_type):
        super().__init__(name, cost)
        self.test_type = test_type

    def calculate_final_bill(self):
        """
        Переопределение бизнес-логики.
        Например, сложные анализы могут иметь скрытые коэффициенты,
        но пока вернем базовую стоимость.
        """
        return self.cost

    def __str__(self):
        base_str = super().__str__()
        return f"{base_str} | Тип: {self.test_type}"

# ==============================================================================
# ЭТАП 1: НАСЛЕДОВАНИЕ (Дочерний класс 2)
# ==============================================================================

class InsuredPatient(Patient):
    """
    Дочерний класс 2: Застрахованный пациент.
    Наследует Patient.
    Специфичное поле: policy_number (номер полиса), discount_rate (процент скидки).
    """

    def __init__(self, full_name, card_number, policy_number, discount_rate=0.20):
        super().__init__(full_name, card_number)
        self.policy_number = policy_number
        self.discount_rate = discount_rate  # По умолчанию скидка 20%

    def __str__(self):
        base_str = super().__str__()
        return f"{base_str} | Полис: {self.policy_number} (Скидка: {int(self.discount_rate * 100)}%)"

# ==============================================================================
# ЭТАП 3 & 4: КОМПОЗИЦИЯ (Класс-контейнер)
# ==============================================================================

class Visit:
    """
    Класс-контейнер: Визит пациента.
    Содержит список услуг и ссылку на пациента.
    """

    def __init__(self, patient: Patient):
        """
        Конструктор принимает объект пациента (любого типа: Patient или InsuredPatient).
        """
        self.patient = patient
        self.services = []  # Список объектов MedicalService (или его наследников)

    def add_item(self, item: MedicalService):
        """Добавляет услугу в список визита."""
        if isinstance(item, MedicalService):
            self.services.append(item)
        else:
            raise TypeError("Можно добавлять только объекты типа MedicalService")

    def get_detailed_report(self):
        """Формирует текстовый отчет по всем услугам."""
        report_lines = []
        report_lines.append(f"\n{'='*50}")
        report_lines.append(f"ОТЧЕТ ПО ВИЗИТУ")
        report_lines.append(f"{'='*50}")
        report_lines.append(f"Пациент: {self.patient.full_name}")

        # Если пациент застрахован, выводим инфо о полисе
        if isinstance(self.patient, InsuredPatient):
            report_lines.append(f"Страховой полис: {self.patient.policy_number}")

        report_lines.append(f"{'-'*50}")
        report_lines.append(f"{'Наименование услуги':<30} | {'Цена':>10}")
        report_lines.append(f"{'-'*50}")

        for service in self.services:
            # Вызываем __str__ у каждой услуги
            report_lines.append(f"{service.name:<30} | {service.cost:>10} руб.")

        return "\n".join(report_lines)

    def calculate_total(self):
        """
        ЭТАП 4: Итоговая логика.
        1. Перебирает список услуг.
        2. Вызывает полиморфный метод calculate_final_bill() у каждой услуги.
        3. Применяет логику Дочернего класса 2 (скидка для InsuredPatient).
        """
        total_sum = 0

        # 1 & 2. Суммируем стоимость услуг через полиморфизм
        for service in self.services:
            # Здесь проявляется полиморфизм: вызывается версия метода
            # конкретного дочернего класса (Consultation или LabTest)
            total_sum += service.calculate_final_bill()

        # 3. Применяем логику субъекта (Пациента)
        final_bill = total_sum

        if isinstance(self.patient, InsuredPatient):
            # Если пациент застрахован, применяем скидку
            discount_amount = total_sum * self.patient.discount_rate
            final_bill -= discount_amount

        return final_bill

# ==============================================================================
# ЭТАП 5: ДЕМОНСТРАЦИЯ СЦЕНАРИЯ (Main)
# ==============================================================================

if __name__ == "__main__":
    print("--- Создание объектов ---")

    # 1. Создаем пациентов (Обычного и Застрахованного)
    regular_patient = Patient("Иванова Мария Петровна", "123-456")
    insured_patient = InsuredPatient("Смирнов Алексей Игоревич", "789-012", "POL-998877", discount_rate=0.25)

    print(regular_patient)
    print(insured_patient)

    # 2. Создаем набор разных услуг (Объекты дочерних классов БК1)
    # Консультация терапевта
    serv_1 = Consultation("Прием терапевта", 2800, "Др. Хаус")
    # Лабораторный анализ крови
    serv_2 = LabTest("Анализ крови (общий)", 1200, "Гематология")
    # УЗИ (тоже консультация/диагностика)
    serv_3 = Consultation("УЗИ сердца", 3500, "Др. Стрэндж")
    # Биохимия
    serv_4 = LabTest("Биохимия печени", 2200, "Биохимия")

    print("\n--- Созданные услуги ---")
    print(serv_1)
    print(serv_2)
    print(serv_3)
    print(serv_4)

    # 3. Сценарий 1: Визит обычного пациента
    print("\n\n>>> СЦЕНАРИЙ 1: Обычный пациент")
    visit_1 = Visit(regular_patient)
    visit_1.add_item(serv_1)
    visit_1.add_item(serv_2)

    print(visit_1.get_detailed_report())
    total_1 = visit_1.calculate_total()
    print(f"{'ИТОГО К ОПЛАТЕ:':<30} | {total_1:>10} руб.")
    print(f"{'='*50}")

    # 4. Сценарий 2: Визит застрахованного пациента (VIP/Insured)
    print("\n\n>>> СЦЕНАРИЙ 2: Застрахованный пациент (со скидкой)")
    visit_2 = Visit(insured_patient)
    # Добавляем больше услуг для наглядности
    visit_2.add_item(serv_1)
    visit_2.add_item(serv_2)
    visit_2.add_item(serv_3)
    visit_2.add_item(serv_4)

    print(visit_2.get_detailed_report())
    total_2 = visit_2.calculate_total()
    print(f"{'ИТОГО К ОПЛАТЕ (со скидкой):':<30} | {total_2:>10.2f} руб.")
    print(f"{'='*50}")

--- Создание объектов ---
Пациент: Иванова Мария Петровна    | Карта №: 123-456
Пациент: Смирнов Алексей Игоревич  | Карта №: 789-012 | Полис: POL-998877 (Скидка: 25%)

--- Созданные услуги ---
Услуга: Прием терапевта      | Базовая цена: 2800 руб. | Врач: Др. Хаус
Услуга: Анализ крови (общий) | Базовая цена: 1200 руб. | Тип: Гематология
Услуга: УЗИ сердца           | Базовая цена: 3500 руб. | Врач: Др. Стрэндж
Услуга: Биохимия печени      | Базовая цена: 2200 руб. | Тип: Биохимия


>>> СЦЕНАРИЙ 1: Обычный пациент

ОТЧЕТ ПО ВИЗИТУ
Пациент: Иванова Мария Петровна
--------------------------------------------------
Наименование услуги            |       Цена
--------------------------------------------------
Прием терапевта                |       2800 руб.
Анализ крови (общий)           |       1200 руб.
ИТОГО К ОПЛАТЕ:                |       4000 руб.


>>> СЦЕНАРИЙ 2: Застрахованный пациент (со скидкой)

ОТЧЕТ ПО ВИЗИТУ
Пациент: Смирнов Алексей Игоревич
Страховой полис: POL-998877
-----